# NeuroDecode–LaBraM — Phase 5: CBraMod + original fast Amazon S3 EEG preparation

**Single Kaggle notebook: build/reuse the exact original 104-subject HDF5, then train the two pretrained CBraMod validation comparators.**

- **No Add Input is required.** If a verified HDF5 already exists in `/kaggle/working`, reuse it; otherwise fetch the **624 original imagery EDF runs** from public `s3://physionet-open/eegmmidb/1.0.0/` and build the same schema-3 HDF5 using the **original pinned S3 builder** and isolated Python 3.11 (MOABB 1.7.2, MNE 1.13.2).
- The 104-subject preprocessing source is pinned to `74e511fb9c1ac4f0543b249b571e28e811ec92c3`. The original S3 builder source is embedded byte-for-byte, not substituted with a new implementation.
- Training begins **only** if SHA-256 matches the previously independently validated HDF5, `f6d2c7824f6aaa522286ea8ee5253b0220602117fc69d7b49c3af6e6a1324f7a`.
- A separate optional **private, single-file Kaggle Dataset** upload runs after verification; upload failure does **not** trigger a rebuild or change the EEG benchmark. Kaggle API credentials/quotas may still apply.
- CBraMod: original pretrained safetensors pinned, same 75 Train / 14 Validation / 15 untouched Test, same seed 42. Neither training mode touches Test waveforms.
- Large virtual environments, raw EDFs and Git clones are kept under **`/kaggle/temp`**, not `/kaggle/working`, to prevent another 13,000-file Notebook Output problem. Only the HDF5, model results, and small reports go into Working outputs.

**Kaggle settings:** Accelerator GPU, Internet On, *do not Add Input from your large old Notebook Output*. Build duration depends on Kaggle S3 throughput and preprocessing: previous successful timing cannot be guaranteed for new sessions.

Scientific source: https://github.com/mohamad679/labram-mi-project/pull/24


In [ ]:
from pathlib import Path
import json, hashlib, subprocess, sys, os
ROOT = Path('/kaggle/working')
REPO = Path('/kaggle/temp/neurodecode_phase5_source')
RESULTS = ROOT/'NeuroDecode_Phase5_CBraMod_Validation'
ZIP = ROOT/'NeuroDecode_Phase5_CBraMod_Validation_Results.zip'
CACHE_OVERRIDE = ''  # Supply full path only if auto-discovery fails.
PINNED_REPO_COMMIT = '561870b2604d2f00198168049164337a5b9e1b16'
PINNED_RUNNER_SHA256 = 'ad14418845d5c16c0a3a9904c6edac7d5a55c4eaa9ed3f1b2ffe6c840ce3f69e'
PINNED_ADAPTER_SHA256 = '672d0cff16b758b95334e81d1b5910c23e7484297698a5d66b296838e34f0cb9'
print('Workdir:',ROOT)
print('Selected source commit:', PINNED_REPO_COMMIT)

# Original Phase-2 EEG source; do not change without re-auditing the entire cache.
S3_PREPROCESS_COMMIT = '74e511fb9c1ac4f0543b249b571e28e811ec92c3'
EEG_H5_SHA256 = 'f6d2c7824f6aaa522286ea8ee5253b0220602117fc69d7b49c3af6e6a1324f7a'
S3_VENV = Path('/kaggle/temp/neurodecode_s3_py311')
S3_SOURCE = Path('/kaggle/temp/neurodecode_s3_source')
S3_MNE = Path('/kaggle/temp/neurodecode_s3_edfs')
S3_RUNNER = Path('/kaggle/temp/phase5_original_fast_s3_builder.py')
H5_WORKING = Path('/kaggle/working/physionet_mi_104subjects.h5')
UPLOAD_PRIVATE_DATASET = True  # Optional; upload can fail independently of training.
PRIVATE_DATASET_OWNER = 'mohamadasgari'  # From your Kaggle Notebook URL; edit if needed.
PRIVATE_DATASET_SLUG = 'neurodecode-physionet-mi104-hdf5-v1'
assert ' ' not in PRIVATE_DATASET_OWNER and ' ' not in PRIVATE_DATASET_SLUG


## A — Prepare/reuse the original 104-subject HDF5 via public Amazon S3

**No Kaggle Notebook Output download and no call to `kaggle kernels output`.** First search the current kernel for the file. If absent, create an isolated Python 3.11 environment outside `/kaggle/working`, fetch 624 EEG EDF runs from the **official public S3 mirror**, process the 104 predefined subjects, and append into one HDF5. The original builder deletes raw EDFs **only after** committing each participant. Interruption can be resumed in the same session by rerunning the **build** cell.

The model-training kernel keeps its own Python and GPU packages. This cell does **not** change the classifier or evaluation protocol.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, hashlib, json

Path('/kaggle/temp').mkdir(parents=True, exist_ok=True)
H5_WORKING.parent.mkdir(parents=True, exist_ok=True)

def sha256_chunks(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(8*1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()

attached = list(Path('/kaggle/input').rglob('physionet_mi_104subjects.h5')) if Path('/kaggle/input').exists() else []
if H5_WORKING.exists():
    CACHE_FOR_EXPERIMENT = H5_WORKING
    print('Existing HDF5 in Working; no S3 rebuild required.')
elif len(attached)==1:
    CACHE_FOR_EXPERIMENT = attached[0]
    print('Existing attached HDF5; no S3 rebuild required.')
elif not attached:
    CACHE_FOR_EXPERIMENT = H5_WORKING
    print('HDF5 absent. Will build directly from the official PhysioNet S3 EDFs.')
else:
    raise RuntimeError('Multiple different HDF5 candidates; remove duplicated inputs to avoid ambiguity.')

free=shutil.disk_usage('/kaggle/working').free / 2**30
print(f'Free /kaggle/working disk: {free:.2f} GiB')
if not CACHE_FOR_EXPERIMENT.exists() and free<5:
    raise RuntimeError('Insufficient disk (<5 GiB) for verified 1.73-GiB HDF5 and model outputs.')


In [ ]:
# Original Python 3.11 MOABB / MNE data processing environment.
# This is isolated from the Python 3.13 GPU model-training kernel.
if not CACHE_FOR_EXPERIMENT.is_file():
    VENV_PY = S3_VENV / 'bin/python'
    setup_log = ROOT / 'phase5_s3_setup.log'
    def setup(*command):
        print('Installing/verifying S3 scientific environment:', command[:3], flush=True)
        with setup_log.open('a', encoding='utf-8') as stream:
            p=subprocess.run(list(map(str,command)),stdout=stream,stderr=subprocess.STDOUT)
        if p.returncode:
            print(setup_log.read_text(errors='replace')[-5500:])
            raise RuntimeError('Isolated scientific environment setup failed; see log')
    uv = shutil.which('uv')
    if not uv:
        setup(sys.executable,'-m','pip','install','--disable-pip-version-check','uv')
        uv = shutil.which('uv') or str(Path(sys.executable).parent/'uv')
    if not VENV_PY.is_file():
        setup(uv,'python','install','3.11')
        setup(uv,'venv','--python','3.11','--seed',str(S3_VENV))
    assert subprocess.check_output([str(VENV_PY),'-c','import sys;print(sys.version_info[:2])'],text=True).strip()=='(3, 11)'
    setup(uv,'pip','install','--python',str(VENV_PY),
          'moabb==1.7.2','mne==1.13.2','boto3>=1.34,<2','h5py>=3.10,<4',
          'numpy>=1.26,<3','scikit-learn>=1.3,<2','tqdm>=4.66,<5',
          'PyYAML>=6,<7','pydantic>=2,<3')
    print('Isolated S3 EEG environment ready (Python 3.11, MOABB 1.7.2, MNE 1.13.2).')
else:
    print('S3 environment installation skipped: HDF5 already exists.')


In [ ]:
# Pin the PREPROCESSING code independently of the Phase-5 model code.
if not CACHE_FOR_EXPERIMENT.is_file():
    REPO_URL = 'https://github.com/mohamad679/labram-mi-project.git'
    if not (S3_SOURCE/'.git').is_dir():
        subprocess.run(['git','clone','--quiet',REPO_URL,str(S3_SOURCE)],check=True)
    subprocess.run(['git','-C',str(S3_SOURCE),'checkout','--detach',S3_PREPROCESS_COMMIT],check=True)
    checked=subprocess.check_output(['git','-C',str(S3_SOURCE),'rev-parse','HEAD'],text=True).strip()
    assert checked == S3_PREPROCESS_COMMIT
    S3_ENV=os.environ.copy()
    S3_ENV.update({'PYTHONPATH':str(S3_SOURCE),'MPLBACKEND':'Agg',
        'MNE_DATA':str(S3_MNE),'MNE_DATASETS_EEGBCI_PATH':str(S3_MNE),
        'MOABB_DOWNLOAD_PROVIDER':'upstream','PYTHONUNBUFFERED':'1'})
    subprocess.run([str(VENV_PY),'-c',
        "import moabb,mne,boto3,h5py; from src.portable_eeg import EXPECTED_SUBJECTS; "
        "assert len(EXPECTED_SUBJECTS)==104; print('Preprocessing versions:',moabb.__version__,mne.__version__)"],
        cwd=S3_SOURCE,env=S3_ENV,check=True)
    print('Source SHA pinned:',checked)
else:
    print('Source checkout skipped: existing HDF5 must still pass SHA verification.')


In [ ]:
# Embed the EXACT previously successful S3->EDF->MOABB->HDF5 builder.
# Do not substitute a different preprocessing script or change its imagery runs.
from pathlib import Path
if not CACHE_FOR_EXPERIMENT.is_file():
    ORIGINAL_FAST_S3_BUILDER = '"""Resumable Kaggle S3 -> MOABB -> portable HDF5 EEG cache.\n\nScientific data processing comes from the separately pinned repository revision.\nThis utility never trains a model or claims performance results.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nimport shutil\nimport sys\nimport time\nfrom pathlib import Path\n\nRUNS = (4, 6, 8, 10, 12, 14)  # Default PhysionetMI imagined runs only\nS3_BUCKET = "physionet-open"\nS3_ROOT = "eegmmidb/1.0.0"\nFILENAME = "physionet_mi_104subjects.h5"\n\n\ndef s3_key(subject: int, run: int) -> str:\n    return f"{S3_ROOT}/S{subject:03d}/S{subject:03d}R{run:02d}.edf"\n\n\ndef edf_path(mne_root: Path, subject: int, run: int) -> Path:\n    return (mne_root / "MNE-eegbci-data" / "files" / "eegmmidb" / "1.0.0"\n            / f"S{subject:03d}" / f"S{subject:03d}R{run:02d}.edf")\n\n\ndef get_client():\n    import boto3\n    from botocore import UNSIGNED\n    from botocore.config import Config\n\n    return boto3.client(\n        "s3", config=Config(signature_version=UNSIGNED,\n                            connect_timeout=20, read_timeout=90,\n                            retries={"mode": "standard", "max_attempts": 8}),\n    )\n\n\ndef download_run(client, subject: int, run: int, mne_root: Path) -> tuple[int, float]:\n    key = s3_key(subject, run)\n    destination = edf_path(mne_root, subject, run)\n    size = int(client.head_object(Bucket=S3_BUCKET, Key=key)["ContentLength"])\n    if size <= 0:\n        raise ValueError(f"S3 returned empty EDF for {key}")\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    if destination.is_file() and destination.stat().st_size == size:\n        return size, 0.0\n    # Download to a temporary file and only promote a complete result.\n    part = destination.with_suffix(destination.suffix + ".partial")\n    part.unlink(missing_ok=True)\n    started = time.monotonic()\n    try:\n        client.download_file(S3_BUCKET, key, str(part))\n        if part.stat().st_size != size:\n            raise IOError(f"Incomplete S3 transfer for {key}: {part.stat().st_size}/{size}")\n        part.replace(destination)\n    finally:\n        part.unlink(missing_ok=True)\n    seconds = max(time.monotonic() - started, 0.001)\n    print(f"  S3 {destination.name}: {size / 2**20:.2f} MiB / {seconds:.1f}s "\n          f"= {size / 2**20 / seconds:.2f} MiB/s", flush=True)\n    return size, seconds\n\n\ndef committed_subjects(path: Path) -> set[int]:\n    import h5py\n    with h5py.File(path, "r") as handle:\n        return set(map(int, json.loads(handle.attrs["completed_subjects"])))\n\n\ndef prepare_output(output: Path, previous: Path | None) -> None:\n    from src.portable_eeg import initialize_cache, inspect_cache, verify_cache_metadata\n\n    output.parent.mkdir(parents=True, exist_ok=True)\n    if previous is not None:\n        previous = previous.resolve()\n        if not previous.is_file():\n            raise FileNotFoundError(f"Previous HDF5 was not attached: {previous}")\n        if previous.suffix.lower() not in (".h5", ".hdf5"):\n            raise ValueError("A previous cache must be a .h5 file")\n        # Validate provenance, but allow a trailing unfinished append: initialize_cache()\n        # will safely truncate it after copying to the writable output.\n        import h5py\n        with h5py.File(previous, "r") as src:\n            verify_cache_metadata(src)\n        if not output.exists():\n            print(f"Restoring earlier saved Kaggle output from {previous}", flush=True)\n            shutil.copy2(previous, output)\n        else:\n            print("Output file already exists; preserving it instead of overwriting with older input.", flush=True)\n    initialize_cache(output)  # Rolls back an interrupted append to last committed subject.\n    inspect_cache(output, require_complete=False)\n\n\ndef checksum(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as src:\n        for chunk in iter(lambda: src.read(8 * 1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef make_cache(output: Path, mne_root: Path, previous: Path | None,\n               max_new_subjects: int, retry_attempts: int, retry_delay: float) -> dict:\n    from src.data import load_physionet_mi\n    from src.portable_eeg import EXPECTED_SUBJECTS, append_subject, inspect_cache\n\n    os.environ["MOABB_DOWNLOAD_PROVIDER"] = "upstream"\n    os.environ["MPLBACKEND"] = "Agg"\n    os.environ["MNE_DATA"] = str(mne_root.resolve())\n    os.environ["MNE_DATASETS_EEGBCI_PATH"] = str(mne_root.resolve())\n    prepare_output(output, previous)\n    done = committed_subjects(output)\n    missing = [sub for sub in EXPECTED_SUBJECTS if sub not in done]\n    print(f"RESUME: {len(done)}/104 subjects committed. "\n          f"This session can add up to {max_new_subjects} subjects.", flush=True)\n    if not missing:\n        print("All eligible subjects already present.", flush=True)\n    client = get_client() if missing and max_new_subjects > 0 else None\n    from botocore.exceptions import BotoCoreError, ClientError\n    started = time.monotonic()\n    processed = 0\n    from tqdm import tqdm\n    with tqdm(total=len(EXPECTED_SUBJECTS), initial=len(done),\n              desc="HDF5 EEG subjects", unit="subject", dynamic_ncols=True, file=sys.stdout) as pbar:\n        for sub in missing:\n            if processed >= max_new_subjects:\n                break\n            t0 = time.monotonic()\n            print(f"\\nSUBJECT {sub:03d}: S3 imagery runs {RUNS}", flush=True)\n            for run in RUNS:\n                last_error = None\n                for attempt in range(retry_attempts + 1):\n                    try:\n                        download_run(client, sub, run, mne_root)\n                        last_error = None\n                        break\n                    except (OSError, BotoCoreError, ClientError) as exc:\n                        # Only transport failures are retried; never retry bad EEG data.\n                        if isinstance(exc, ClientError):\n                            code = str(exc.response.get("Error", {}).get("Code", ""))\n                            if code in {"403", "404", "AccessDenied", "NoSuchKey"}:\n                                raise\n                        last_error = exc\n                        if attempt >= retry_attempts:\n                            raise RuntimeError(f"S3 failed for {s3_key(sub, run)} "\n                                               f"after {attempt + 1} attempts") from exc\n                        delay = min(retry_delay * 2**attempt, 120)\n                        print(f"  S3 RETRY {attempt+1}/{retry_attempts} after "\n                              f"{type(exc).__name__}: sleeping {delay:.0f}s", flush=True)\n                        time.sleep(delay)\n            X, y, metadata = load_physionet_mi(subjects=[sub])\n            observed = set(map(int, metadata["subject"].to_numpy()))\n            if observed != {sub}:\n                raise ValueError(f"MOABB returned wrong subject: expected {sub}, got {observed}")\n            if X.shape[1:] != (64, 400):\n                raise ValueError(f"MOABB EEG shape mismatch: {X.shape}")\n            append_subject(output, sub, X, y)\n            del X, y, metadata\n            # Do not delete a raw EDF until HDF5 commit is verified.\n            if sub not in committed_subjects(output):\n                raise RuntimeError(f"HDF5 commit missing after subject {sub}")\n            for run in RUNS:\n                edf_path(mne_root, sub, run).unlink(missing_ok=True)\n            processed += 1\n            pbar.update(1)\n            total_elapsed = time.monotonic() - started\n            eta = total_elapsed / processed * max(0, len(missing) - processed) / 60\n            pbar.set_postfix_str(f"last={sub}, ETA~{eta:.0f}min, file={output.stat().st_size / 2**30:.2f}GiB")\n            print(f"SAVED subject {sub:03d} in {time.monotonic() - t0:.0f}s; "\n                  f"{len(done) + processed}/104 committed", flush=True)\n    y, subjects, shape = inspect_cache(output, require_complete=False)\n    complete = len(set(map(int, subjects))) == len(EXPECTED_SUBJECTS)\n    if complete:\n        inspect_cache(output, require_complete=True)\n    result = {\n        "status": "complete" if complete else "partial",\n        "subjects": int(len(set(map(int, subjects)))),\n        "epochs": int(len(y)),\n        "shape": list(map(int, shape)),\n        "file": str(output),\n        "bytes": output.stat().st_size,\n        "source_commit": os.environ.get("SOURCE_COMMIT", "unknown"),\n        "s3_bucket": S3_BUCKET,\n        "s3_key_prefix": S3_ROOT,\n        "imagined_runs": list(RUNS),\n        "sha256": checksum(output),\n    }\n    report_file = output.with_name("cache_build_report.json")\n    report_file.write_text(json.dumps(result, indent=2) + "\\n", encoding="utf-8")\n    print("\\nCACHE RESULT:", json.dumps(result, indent=2), flush=True)\n    if not complete:\n        print("PARTIAL FILE: Save Kaggle Notebook output and attach it as input in the "\n              "next session. /kaggle/working alone is NOT a permanent backup!", flush=True)\n    else:\n        print("COMPLETE VERIFIED CACHE: create a PRIVATE Kaggle Dataset from this "\n              "Notebook\'s output, then attach it to run_kaggle.ipynb.", flush=True)\n    return result\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__)\n    ap.add_argument("--output", required=True, type=Path)\n    ap.add_argument("--probe", action="store_true", help="Time a single S3 EDF without building HDF5")\n    ap.add_argument("--mne-root", required=True, type=Path)\n    ap.add_argument("--previous", default="", help="Optional attached previous partial HDF5 from /kaggle/input")\n    ap.add_argument("--max-new-subjects", type=int, default=104)\n    ap.add_argument("--retry-attempts", type=int, default=5)\n    ap.add_argument("--retry-delay", type=float, default=8.0)\n    args = ap.parse_args()\n    if not 0 <= args.max_new_subjects <= 104 or args.retry_attempts < 0 or args.retry_delay < 0:\n        ap.error("invalid count or retry settings")\n    if args.probe:\n        size, seconds = download_run(get_client(), 1, 14, args.mne_root)\n        if seconds == 0:\n            # The probe file may already exist from a previous test.\n            print("S3 PROBE: cached file exists; remove it before measuring again.", flush=True)\n        else:\n            print(f"S3 PROBE: {size / 2**20 / seconds:.2f} MiB/s "\n                  f"({8 * size / 1e6 / seconds:.2f} Mbps)", flush=True)\n        return\n    make_cache(args.output, args.mne_root, Path(args.previous) if args.previous else None,\n               args.max_new_subjects, args.retry_attempts, args.retry_delay)\n\n\nif __name__ == "__main__":\n    main()\n'
    S3_RUNNER.write_text(ORIGINAL_FAST_S3_BUILDER,encoding='utf-8')
    subprocess.run([str(VENV_PY),'-m','py_compile',str(S3_RUNNER)],check=True)
    print('Original verbatim S3 builder ready:', S3_RUNNER)
    print('Original builder SHA256:', hashlib.sha256(ORIGINAL_FAST_S3_BUILDER.encode()).hexdigest())
else:
    print('Builder creation skipped: existing HDF5.')


In [ ]:
# BUILD or RESUME the SAME frozen 104-participant HDF5.
# If interrupted, rerun this cell before the Kaggle session ends.
if not CACHE_FOR_EXPERIMENT.is_file():
    cmd=[str(VENV_PY),'-u',str(S3_RUNNER),
        '--output',str(H5_WORKING),'--mne-root',str(S3_MNE),
        '--max-new-subjects','104','--retry-attempts','5','--retry-delay','8']
    print('Building original EEG HDF5 from public Amazon S3; never training on Test.',flush=True)
    log=ROOT/'phase5_s3_build.log'
    with log.open('a',encoding='utf-8') as stream:
        proc=subprocess.Popen(cmd,cwd=S3_SOURCE,env=S3_ENV,stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout:
            print(line,end='',flush=True)
            stream.write(line)
        rc=proc.wait()
    if rc:
        raise RuntimeError(f'S3 HDF5 builder failed (exit {rc}). Existing subject commits can resume in this session.')
assert CACHE_FOR_EXPERIMENT.is_file(), 'S3 builder did not complete the cache.'
actual=sha256_chunks(CACHE_FOR_EXPERIMENT)
print('HDF5 file:',CACHE_FOR_EXPERIMENT)
print('HDF5 size GiB:',round(CACHE_FOR_EXPERIMENT.stat().st_size/2**30,3))
print('HDF5 SHA256:',actual)
assert actual==EEG_H5_SHA256, (
    'FULL CACHE HASH MISMATCH: fail closed. Do not train; send build report/log '
    'so source conversion/environment can be compared to original accepted snapshot.'
)
CACHE_OVERRIDE = str(CACHE_FOR_EXPERIMENT)
print('SCIENTIFIC HDF5 CONTRACT PASS — Training is authorized on Train/Validation only.')


## B — Optional: publish ONLY this HDF5 as a PRIVATE Kaggle Dataset

**This step never retrieves an old Notebook Output** and does not upload Git clones, virtual environments, EDFs, ZIP results or checkpoints. It creates a dedicated upload folder with one HDF5 hard link (no second 1.73 GiB local copy) plus a small `dataset-metadata.json` file. Kaggle CLI's `datasets create` is **private by default when `--public` is omitted**.

This is a **one-time upload** to Kaggle's Dataset storage; unlike a local hard link, it requires an actual network transfer, and Kaggle may enforce API credentials or rate limits. A failure here is reported, **without rebuilding HDF5 or preventing training**. After a success, independently confirm it appears under *Your Datasets* with Private visibility. Once private, subsequent notebooks can use Add Input → Your Datasets to mount this **one-file** Dataset.

In [ ]:
# Secure, one-file upload attempt; NEVER use `kaggle kernels output`.
# Any API failure is independent of the scientifically audited training file.
from pathlib import Path
import json, os, subprocess, sys

if UPLOAD_PRIVATE_DATASET:
    source = Path(CACHE_OVERRIDE).resolve()
    assert source.is_file() and sha256_chunks(source)==EEG_H5_SHA256
    if source.parent != ROOT:
        print('HDF5 was already mounted from an input. Private upload skipped (avoid duplicating uploaded data).')
    else:
        upload_folder=ROOT/'phase5_private_hdf5_only'
        upload_folder.mkdir(exist_ok=True)
        link=upload_folder/'physionet_mi_104subjects.h5'
        if link.exists():
            assert os.path.samefile(link,source),'Upload folder points to a different HDF5!'
        else:
            # Same /kaggle/working filesystem -> zero extra HDF5 disk allocation.
            os.link(source,link)
        metadata={
            'title':'NeuroDecode PhysioNet MI 104 Subjects HDF5',
            'id':f'{PRIVATE_DATASET_OWNER}/{PRIVATE_DATASET_SLUG}',
            'licenses':[{'name':'other'}],
            'description':(
              'PRIVATE research HDF5 cache reconstructed from the PhysioNet EEG Motor '
              'Movement/Imagery Dataset v1.0.0 (DOI: 10.13026/C28G6P); '
              'upstream Open Data Commons Attribution 1.0 conditions apply. '
              'MOABB 1.7.2 and MNE 1.13.2, six imagery runs and fixed 104-subject cohort. '
              'Scientific HDF5 SHA-256: '+EEG_H5_SHA256+'.')}
        (upload_folder/'dataset-metadata.json').write_text(
            json.dumps(metadata,indent=2)+'\n',encoding='utf-8')
        assert {p.name for p in upload_folder.iterdir()}=={
            'physionet_mi_104subjects.h5','dataset-metadata.json'}
        print('PRIVATE DATASET upload folder has exactly one HDF5 + metadata.')
        print('Dataset ID:',metadata['id'])
        print('Attempting one Kaggle API upload; no --public flag; do not rerun on rate limit.')
        try:
            # Kaggle CLI is installed in Kaggle kernels. Do NOT print tokens/credentials.
            subprocess.run([sys.executable,'-m','pip','install','-q','kaggle'],check=True)
            completed=subprocess.run([
                'kaggle','datasets','create','-p',str(upload_folder),
                '--keep-tabular','--dir-mode','skip'],
                capture_output=True,text=True,timeout=900)
            if completed.returncode:
                print('PRIVATE DATASET UPLOAD UNCONFIRMED. Kaggle API reported:',
                    (completed.stderr or completed.stdout)[-900:])
                print('Training may proceed using the validated local HDF5. '
                      'Do not retry repeatedly after HTTP 429.')
            else:
                print('Kaggle API accepted private Dataset upload request.')
                print('Check Private visibility and verify one HDF5 in Your Datasets.')
                print('https://www.kaggle.com/datasets/'+metadata['id'])
        except (subprocess.CalledProcessError,subprocess.TimeoutExpired,OSError) as exc:
            print('Private Dataset upload did not complete:',type(exc).__name__)
            print('Validated local HDF5 remains intact. Continue training; save output version.')
        finally:
            # Do not leave a second 1.73-GiB hardlink in Notebook Output.
            # Removing this alias cannot delete the actual HDF5 source.
            if link.is_file() and os.path.samefile(link,source):
                link.unlink()
                print('Temporary HDF5 upload alias removed. Original verified HDF5 preserved.')
else:
    print('Private Dataset publishing disabled; local HDF5 is still verified.')


In [ ]:
import subprocess, sys
pkgs=['braindecode[hub]==1.5.1','mne==1.13.2','scipy>=1.14,<2',
      'h5py>=3.10,<5','scikit-learn>=1.6,<2','numpy>=2,<3',
      'safetensors>=0.4,<1','huggingface_hub>=0.30,<2']
subprocess.run([sys.executable,'-m','pip','install','--quiet',*pkgs],check=True)
import torch, braindecode, scipy, numpy as np, h5py, sklearn
print({'python':sys.version.split()[0], 'torch':torch.__version__,
       'CUDA':torch.cuda.is_available(), 'braindecode':braindecode.__version__,
       'scipy':scipy.__version__, 'numpy':np.__version__})
assert torch.cuda.is_available(), 'GPU is required: set Kaggle Accelerator=GPU before this study'


In [ ]:
import subprocess, hashlib
# Reuse only an already present, hash-verified source checkout on reruns.
# Source lives in /kaggle/temp so its thousands of files are excluded from Notebook Output.
if not REPO.exists():
    subprocess.run(['git','clone','--quiet','--branch','feat/phase5-cbramod-foundation-comparator-20261013',
                    '--single-branch','https://github.com/mohamad679/labram-mi-project.git',str(REPO)],check=True)
else:
    assert (REPO/'.git').is_dir(), 'Source folder already exists but is not a Git checkout'
subprocess.run(['git','-C',str(REPO),'checkout','--quiet',PINNED_REPO_COMMIT],check=True)
head=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert head==PINNED_REPO_COMMIT
sha=lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(REPO/'scripts/run_phase5_cbramod.py')==PINNED_RUNNER_SHA256, 'Experiment source mismatch'
assert sha(REPO/'src/phase5_cbramod.py')==PINNED_ADAPTER_SHA256, 'Input adapter source mismatch'
print('Source and adapter SHA-256 verified:',head)


In [ ]:
from pathlib import Path
candidates=[Path(CACHE_OVERRIDE)] if CACHE_OVERRIDE else (
    list(Path('/kaggle/input').rglob('physionet_mi_104subjects.h5'))+
    [Path('/kaggle/working/physionet_mi_104subjects.h5')])
candidates=[p for p in candidates if p.is_file()]
assert len(candidates)==1, ('Must locate exactly one HDF5. Found:',list(map(str,candidates)))
CACHE=candidates[0]
expected_sha='f6d2c7824f6aaa522286ea8ee5253b0220602117fc69d7b49c3af6e6a1324f7a'
assert sha(CACHE)==expected_sha, 'HDF5 dataset SHA mismatch — do not proceed'
print('Valid complete prior 104-subject HDF5:', CACHE, round(CACHE.stat().st_size/2**30,3),'GiB')


## Pre-training compatibility gate

Before any expensive full-cohort training, this runs an actual **strict pretrained safetensors** load, checks SHA and runs a shape/finite-value forward pass on a *Train-only* EEG epoch. Failure aborts rather than silently training a different architecture.

In [ ]:
import sys
sys.path.insert(0,str(REPO))
from scripts.run_phase5_cbramod import load_checkpoint_from_hf, validate_frozen_split
from src.phase5_cbramod import load_pinned_cbramod, adapt_cbramod_input
from src.portable_eeg import inspect_cache, H5SignalSource, LABELS
labels,subjects,shape=inspect_cache(CACHE, require_complete=True)
y=np.searchsorted(np.asarray(LABELS),labels)
manifest=json.loads((REPO/'configs/phase3_locked_split.json').read_text())
split=validate_frozen_split(y,subjects,manifest)
assert len(split['train'])==13050 and len(split['validation'])==2436 and len(split['test'])==2610
WEIGHTS=load_checkpoint_from_hf()
model,pretrained_info=load_pinned_cbramod(WEIGHTS,device='cpu')
source=H5SignalSource(CACHE,shape)
try:
    selected=int(split['train'][0])
    assert int(subjects[selected]) in manifest['partitions']['train']['subject_ids']
    wave=adapt_cbramod_input(source[selected])
finally:
    source.close()
with torch.inference_mode():
    logits=model(torch.from_numpy(wave[None]))
assert tuple(logits.shape)==(1,5) and bool(torch.isfinite(logits).all())
del model
print('PREFLIGHT PASS. True pretrained CBraMod loaded and train-only forward succeeded.')
print(json.dumps(pretrained_info,indent=2))


## Execute both frozen-backbone and full-backbone runs

Ten-epoch cap, 3-epoch early stopping patience, train-only class weights, head LR 0.001, backbone LR 0.00003, no Test inference. If Kaggle GPU OOM occurs, share the error; **do not silently alter hyperparameters and combine incompatible runs**.

In [ ]:
import subprocess, sys
assert not RESULTS.exists() and not ZIP.exists(), 'Results already exist; do not overwrite a completed scientific run'
cmd=[sys.executable,'-m','scripts.run_phase5_cbramod',
     '--cache',str(CACHE),'--split-manifest',str(REPO/'configs/phase3_locked_split.json'),
     '--output-dir',str(RESULTS),'--modes','probe','full','--device','cuda',
     '--seed','42','--epochs','10','--patience','3','--batch-size','8','--bootstrap','500','--amp']
logfile=ROOT/'NeuroDecode_Phase5_CBraMod_execution.log'
with logfile.open('w',encoding='utf-8') as log:
    process=subprocess.Popen(cmd,cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                             text=True,bufsize=1)
    for line in process.stdout:
        print(line,end='')
        log.write(line)
        log.flush()
    code=process.wait()
if code:
    raise RuntimeError(f'Phase 5 runner FAILED, exit={code}; share execution log, do not report scientific PASS')
print('COMPLETE: both genuine pretrained CBraMod Validation runs finished')


In [ ]:
import zipfile, datetime
assert (RESULTS/'phase5_summary.json').is_file()
with zipfile.ZipFile(ZIP,'x',compression=zipfile.ZIP_DEFLATED,compresslevel=7) as z:
    z.write(RESULTS/'phase5_summary.json',arcname='phase5_summary.json')
    z.write(logfile,arcname='phase5_execution.log')
    for mode in ('probe','full'):
        for name in ('report.json','validation_predictions.npz'):
            f=RESULTS/mode/name
            assert f.is_file(), str(f)
            z.write(f,arcname=f'{mode}/{name}')
    z.writestr('run_provenance.json',json.dumps({
        'generated_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'dataset_sha256':expected_sha,'runner_sha256':PINNED_RUNNER_SHA256,
        'adapter_sha256':PINNED_ADAPTER_SHA256,'source_commit':PINNED_REPO_COMMIT,
        'validation_only':True, 'test_set_locked':True,
    },indent=2))
with zipfile.ZipFile(ZIP) as z:
    assert z.testzip() is None
    assert all(not x.endswith(('.pt','.safetensors','.h5')) for x in z.namelist())
print('VALIDATION-ONLY ZIP READY:', ZIP)
print('ZIP bytes:', ZIP.stat().st_size,'SHA256:',sha(ZIP))


## Download and scientific handling

The ZIP deliberately excludes all EEG and large trained model weights; **save a Kaggle version of the notebook Output** if you wish to preserve exact checkpoint binaries. Send the small ZIP back for independent validation. Do not use the frozen Test split for model selection.

In [ ]:
from IPython.display import display, HTML
import base64
if ZIP.is_file():
    payload=base64.b64encode(ZIP.read_bytes()).decode('ascii')
    display(HTML(f'<a download="{ZIP.name}" href="data:application/zip;base64,{payload}" '
                 'style="display:inline-block;background:#2762a9;color:white;padding:12px;'
                 'border-radius:7px;text-decoration:none;font-weight:bold">'
                 'Download Phase 5 Results ZIP</a>'))
    print('If Safari blocks this link, refresh Kaggle Output /kaggle/working and use Download.')
else:
    print('No ZIP: previous training step failed or was not run.')
